# Agent-Harness Benchmark – Auswertung
Dieses Notebook bewertet fünf Sprachmodelle eines MDE-Agent-Harness (bxAgent).
Jedes Modell wurde mit derselben Transformation-Aufgabe (Families → Persons) **zehnmal** ausgeführt.
**Ausgewertete Metriken:**
- **Response Latency** – Gesamtlaufzeit pro Durchlauf aus LangFuse-Trace-Latenz
- **Token-Effizienz** – Nicht verfügbar (siehe Abschnitt *Datenquellen & Annahmen*)
- **Task Completion Rate** – Erfolgsquote je Modell, basierend auf dem `success`-Feld der Ergebnis-JSONs
> **Hinweis:** Die Auswertung basiert auf jeweils 10 Wiederholungen pro Modell. Die Ergebnisse sind daher als **erste Eindrücke** zu verstehen, nicht als statistisch signifikante Vergleiche.

## 1. Konfiguration & Pfadzuordnungen
Alle Dateipfade, Feldzuordnungen und Modell-Kürzel sind hier zentral definiert.
**Wichtig:** Dieses Notebook muss aus dem Projektverzeichnis ausgeführt werden (d. h. `.mdeagent-benchmark/` muss im aktuellen Verzeichnis erreichbar sein).

In [ ]:
from pathlib import Path
import os

# ── Dateipfade – auflösen vom aktuellen Arbeitsverzeichnis aus ──────────────────────
# Das Notebook sollte aus dem Projektverzeichnis ausgeführt werden.
# Falls das nicht der Fall ist, wird das Verzeichnis der Notebook-Datei als Basis verwendet.
try:
    NOTEBOOK_DIR = Path(os.getcwd()).resolve()
    # Prüfe, ob wir im Projektverzeichnis sind
    if not (NOTEBOOK_DIR / ".mdeagent-benchmark").exists():
        # Versuche das Verzeichnis der Notebook-Datei zu finden
        try:
            import IPython
            NOTEBOOK_DIR = Path(IPython.get_ipython().user_ns['__file__']).parent.resolve()
        except Exception:
            # Fallback: ein Verzeichnis höher suchen
            NOTEBOOK_DIR = NOTEBOOK_DIR.parent
            if not (NOTEBOOK_DIR / ".mdeagent-benchmark").exists():
                NOTEBOOK_DIR = NOTEBOOK_DIR.parent
except Exception:
    NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR
RESULTS_DIR = PROJECT_ROOT / ".mdeagent-benchmark" / "results"

# ── Feldzuordnungen in den Ergebnis-JSONs ──────────────────────────────
FIELD_MODEL_ID = "model_id"          # Modell-ID
FIELD_ITERATION = "iteration"         # Iterationsnummer (1-10)
FIELD_SUCCESS = "success"             # Boolesch: True = Task erfolgreich
FIELD_TIMESTAMP = "timestamp"         # ISO 8601-Zeitstempel des Starts
FIELD_ERROR = "error"                 # Dict mit Fehlerinfo oder None
FIELD_TRACE_ID = "trace_id"           # LangFuse trace_id (oft None)

# ── LangFuse-Zugang ─────────────────────────────────────────────────────────
LANGFUSE_PUBLIC_KEY = os.environ.get(
    "LANGFUSE_PUBLIC_KEY", "pk-lf-aa4250ae-73cf-49bf-9b29-fdae2aeee2ed"
)
LANGFUSE_SECRET_KEY = os.environ.get(
    "LANGFUSE_SECRET_KEY", "sk-lf-64368355-0dc4-44c5-aaf2-85c9f90bbac9"
)
LANGFUSE_BASE_URL = os.environ.get(
    "LANGFUSE_BASE_URL", "http://localhost:3000"
)

# ── Modell-Kürzel für die Darstellung ──────────────────────────────────────────
MODEL_SHORTCUTS = {
    "gpt-oss-120b": "GPT-OSS 120B",
    "google/gemma-4-E4B-it": "Gemma 4 E4B",
    "Qwen/Qwen3.8-27B-FP8": "Qwen 3.8 27B",
    "GaleneAI/Magistral-Small-2509-FP8-Dynamic": "Magistral Small",
    "RedHatAI/gemma-4-31B-it-FP8-block": "gemma-4 31B (RedHat)",
}

print("Pfad-Konfiguration:")
print(f"  Projektroot:  {PROJECT_ROOT}")
print(f"  Ergebnisse:   {RESULTS_DIR} (existiert={RESULTS_DIR.exists()})")
print(f"  LangFuse:     {LANGFUSE_BASE_URL}")
print(f"  Modelle:      {len(MODEL_SHORTCUTS)}")
print(f"  Ergebnis-Dateien: {len(list(RESULTS_DIR.glob('*.json')))}")


## 2. Datenquellen & Annahmen
### Verfügbare Daten
| Quelle | Felder | Umfang |
|--------|--------|--------|
| Ergebnis-JSONs (`results_dir/*.json`) | `model_id`, `iteration`, `success`, `timestamp`, `error`, `trace_id`, `iterations_completed`, `evaluation_runs`, `written_files` | 50 Dateien (5 Modelle × 10 Iterationen) |
| LangFuse API | `trace.latency`, `trace.observations[].latency` | 100 Traces (jedes Modell/Iteration) |
### Response Latency
Die **Response Latency** wird aus der LangFuse-Trace-Latenz (`trace.latency` in Sekunden) abgeleitet.
Dies ist die gemessene Gesamtdauer vom Start des Traces bis zum Abschluss – also vom ersten Token der Anfrage bis zum letzten Ergebnis des Agent-Laufs.
Die Trace-Namen folgen dem Muster:
```
MDEAgent-Benchmark-{model_id_with_dashes}-iter{N}
```
Beispiel: `MDEAgent-Benchmark-gpt-oss-120b-iter3`
### Token-Effizienz
**Token-Verbrauch ist in den vorliegenden Daten nicht verfügbar.**
Weder die Ergebnis-JSONs enthalten Token-Felder, noch liefern die LangFuse-Observations `input_count`, `output_count` oder `total_count`.  
Die LangFuse-Observations speichern nur `latency` und `time_to_first_token` (beide `None`).
Mögliche Gründe:
- Die LangFuse-Integration protokolliert Token-Zählungen nicht (kein `usage`-Attribut in den Observations).
- Die API-Responses wurden ohne Token-Metadaten an LangFuse gesendet.
### Task Completion Rate
Der Erfolg wird ausschließlich über das `success`-Feld in den Ergebnis-JSONs bestimmt (`True` = erfolgreich, `False` = fehlgeschlagen).
Fehlerursachen umfassen:
- `RuntimeError: Maximum iteration count reached` (Agent konnte die Aufgabe in der maximalen Iterationszahl nicht lösen)
- `Request timed out` (API-Antwort zeitüberschritten)
- `ContextWindowExceededError` (Eingabetext das Kontextfenster des Modells überschreitet)
### Wilson-Konfidenzintervalle
Für die Completion Rate werden **Wilson-Score-Konfidenzintervalle** (95 %) berechnet.  
Diese sind auch bei kleinen Stichproben (n=10) und extremen Raten (0 % oder 100 %) zuverlässiger als die einfache Normalapproximation.

## 3. Daten laden und bereinigen

In [ ]:
import json
import pandas as pd
import numpy as np

# Ergebnis-JSONs laden
records = []
missing_files = []
for json_file in sorted(RESULTS_DIR.glob("*.json")):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        data["source_file"] = json_file.stem
        records.append(data)
    except Exception as e:
        missing_files.append((json_file.stem, str(e)))

if missing_files:
    print("Fehlgeschlagene Lesevorgänge:")
    for fname, err in missing_files:
        print(f"  {fname}: {err}")

# DataFrame erstellen
df = pd.DataFrame(records)
df = df.sort_values([FIELD_MODEL_ID, FIELD_ITERATION]).reset_index(drop=True)

# Modell-Kürzel hinzufügen
df["model_short"] = df[FIELD_MODEL_ID].map(MODEL_SHORTCUTS).fillna(df[FIELD_MODEL_ID])

# Zeitstempel parsen
df["timestamp_dt"] = pd.to_datetime(df[FIELD_TIMESTAMP], utc=True)

# Fehlende Werte melden
print(f"Gesamtzahl Läufe: {len(df)}")
print(f"Einzigartige Modelle: {df[FIELD_MODEL_ID].nunique()}")
print(f"\nLäufe pro Modell:")
print(df.groupby("model_short").size().to_string())
print(f"\nErfolgsraten pro Modell:")
success_rate = (df.groupby("model_short")[FIELD_SUCCESS].sum() / df.groupby("model_short").size() * 100)
for model, rate in success_rate.items():
    print(f"  {model}: {rate:.1f}%")

# Daten auf relevante Spalten reduzieren
df_bench = df[[
    "model_id", "model_short", FIELD_ITERATION, FIELD_SUCCESS,
    FIELD_TIMESTAMP, "timestamp_dt", "source_file"
]].copy()

print(f"\nRelevante Spalten: {list(df_bench.columns)}")
print("\nErste Zeilen:")
print(df_bench.head(10).to_string(index=False))


## 4. LangFuse-Trace-Daten laden
Die Response Latency wird aus LangFuse-Traces abgeleitet.

In [ ]:
from langfuse import Langfuse

# LangFuse-Client verbinden
lf_client = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_BASE_URL,
)

print(f"LangFuse verbunden ✓  Host: {LANGFUSE_BASE_URL}")

# Trace-Namen-Muster aus dem Benchmark-Code:
# MDEAgent-Benchmark-{model_id_with_dashes}-iter{N}
# Beispiel: MDEAgent-Benchmark-gpt-oss-120b-iter3
TRACE_PREFIX = "MDEAgent-Benchmark-"
TRACE_SUFFIX = "-iter"

def build_trace_name(model_id: str, iteration: int) -> str:
    """Konstruiere den erwarteten LangFuse-Trace-Namen."""
    return f"{TRACE_PREFIX}{model_id.replace('/', '-')}{TRACE_SUFFIX}{iteration}"

print("LangFuse-Client und Trace-Namen-Helfer bereit.")


In [ ]:
# Alle Traces abrufen und in ein Dict mappen
# LangFuse API erlaubt max. limit=100; daher seitenweise laden
trace_data = {}  # {trace_name: latency}
all_traces = []
page = 1
page_size = 100
while True:
    batch = lf_client.api.trace.list(page=page, limit=page_size)
    if not batch.data:
        break
    all_traces.extend(batch.data)
    if len(batch.data) < page_size:
        break
    page += 1

print(f"Anzahl Traces in LangFuse: {len(all_traces)}")

if all_traces:
    for t in all_traces:
        trace_data[t.name] = t.latency
    
    # Zeige alle gefundenen Trace-Namen
    print("\nAlle Trace-Namen:")
    for name in sorted(trace_data.keys()):
        print(f"  {name} \u2192 {trace_data[name]:.1f}s")

    # Prüfe, wie viele Benchmark-Ergebnisse einen Trace haben
    df_bench["trace_name"] = df_bench.apply(
        lambda r: build_trace_name(r["model_id"], int(r[FIELD_ITERATION])), axis=1
    )
    df_bench["trace_found"] = df_bench["trace_name"].isin(trace_data.keys())
    df_bench["trace_latency_s"] = df_bench["trace_name"].map(trace_data)
    
    found_count = df_bench["trace_found"].sum()
    print(f"\nTraces gefunden: {found_count}/{len(df_bench)} ({found_count/len(df_bench)*100:.0f}%)")
else:
    print("Keine Traces in LangFuse gefunden.")
    df_bench["trace_found"] = False
    df_bench["trace_latency_s"] = np.nan


## 5. Datenqualität prüfen

In [ ]:
print("=" * 60)
print("Datenqualitäts-Check")
print("=" * 60)

# 1. Vollständigkeit
print(f"\nGesamtzahl Läufe: {len(df_bench)}")
print(f"Erwartet: {len(MODEL_SHORTCUTS) * 10}")

# 2. Fehlende Latenz
missing_latency = df_bench["trace_latency_s"].isna().sum()
print(f"\nFehlende Latenz-Daten: {missing_latency}/{len(df_bench)}")
if missing_latency > 0:
    missing_runs = df_bench[df_bench["trace_latency_s"].isna()]
    for _, row in missing_runs.iterrows():
        print(f"  - {row['model_short']} Iter {row['iteration']}: '{row['trace_name']}'")

# 3. Duplikate prüfen
dupes = df_bench.duplicated(subset=["model_id", FIELD_ITERATION])
print(f"\nDuplikate (modell+iteration): {dupes.sum()}")

# 4. Iterationslücken
print(f"\nIterationen pro Modell:")
for model, group in df_bench.groupby("model_short"):
    iters = sorted(group[FIELD_ITERATION].astype(int).tolist())
    expected = list(range(1, 11))
    missing_iters = set(expected) - set(iters)
    extra_iters = set(iters) - set(expected)
    status = "OK" if not missing_iters and not extra_iters else "WARN"
    print(f"  {status} {model}: {iters}")
    if missing_iters:
        print(f"      Fehlende Iterationen: {missing_iters}")
    if extra_iters:
        print(f"      Unerwartete Iterationen: {extra_iters}")

# 5. Zusammenfassung
n_with_latency = len(df_bench[df_bench["trace_latency_s"].notna()])
print(f"\nDaten für Auswertung bereit: {n_with_latency} Läufe mit Latenz")


## 6. Aggregierte Kennzahlen

In [ ]:
from scipy import stats

# Nur Läufe mit Latenz-Daten
df_with_latency = df_bench[df_bench["trace_latency_s"].notna()].copy()

summary_rows = []
for model_short, group in df_with_latency.groupby("model_short"):
    latencies = group["trace_latency_s"]
    successes = group[FIELD_SUCCESS].sum()
    total = len(group)
    
    summary_rows.append({
        "Modell": model_short,
        "Anzahl": int(total),
        "Erfolge": int(successes),
        "Erfolgsrate": f"{successes/total*100:.1f}%",
        "Latenz (Median)": f"{latencies.median():.1f}s",
        "Latenz (Mean ± σ)": f"{latencies.mean():.1f}s ± {latencies.std():.1f}s",
        "Latenz (Min–Max)": f"{latencies.min():.1f}s – {latencies.max():.1f}s",
        "IQR": f"{stats.iqr(latencies):.1f}s",
    })

# Token-Info als Hinweis
summary_rows.append({
    "Modell": "───",
    "Anzahl": "–",
    "Erfolge": "–",
    "Erfolgsrate": "–",
    "Latenz (Median)": "–",
    "Latenz (Mean ± σ)": "–",
    "Latenz (Min–Max)": "–",
    "IQR": "–",
})
summary_rows.append({
    "Modell": "Token-Verbrauch",
    "Anzahl": "",
    "Erfolge": "",
    "Erfolgsrate": "",
    "Latenz (Median)": "Nicht verfügbar",
    "Latenz (Mean ± σ)": "Nicht verfügbar",
    "Latenz (Min–Max)": "Nicht verfügbar",
    "IQR": "Nicht verfügbar",
})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print(f"\nHinweis: Token-Verbrauch ist in den vorliegenden Daten (JSONs + LangFuse) nicht gespeichert.")


## 7. Diagramm: Response Latency
Boxplot der LangFuse-Trace-Latenz pro Modell mit overplottender der Einzelmessungen.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Nicht-interaktiver Backend für Notebook-Ausführung
import matplotlib.pyplot as plt
import seaborn as sns

matplotlib.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
})

fig, ax = plt.subplots(figsize=(12, 7))

# Boxplot
palette = sns.color_palette("husl", len(df_with_latency["model_short"].unique()))
box = sns.boxplot(
    x="model_short",
    y="trace_latency_s",
    data=df_with_latency,
    palette=palette,
    width=0.5,
    fliersize=0,
    ax=ax,
)

# Einzelmessungen overplotten
sns.swarmplot(
    x="model_short",
    y="trace_latency_s",
    data=df_with_latency,
    color="black",
    size=6,
    alpha=0.6,
    ax=ax,
)

# Median-Werte annotieren
for i, model in enumerate(df_with_latency["model_short"].unique()):
    med = df_with_latency[df_with_latency["model_short"] == model]["trace_latency_s"].median()
    ax.text(i, med + 5, f"p50={med:.0f}s", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_title("Response Latency pro Modell (LangFuse-Trace-Latenz)", pad=20)
ax.set_xlabel("Modell")
ax.set_ylabel("Latenz [s]")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("benchmark_latency.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Diagramm gespeichert: benchmark_latency.png")


## 8. Token-Verbrauch
**Nicht verfügbar.** Token-Daten sind in den Ergebnis-JSONs sowie in den LangFuse-Observations nicht gespeichert.
Die LangFuse-Observations enthalten nur `latency` und `time_to_first_token` (beide `None`). Keine der folgenden Eigenschaften ist vorhanden:
- `input_count`
- `output_count`
- `total_count`
- `usage`-Objekt
Ein Boxplot kann hier nicht erstellt werden.

In [ ]:
print("=" * 60)
print("Token-Verbrauch – Status")
print("=" * 60)
print()
print("Token-Daten NICHT verfügbar.")
print()
print("Überprüfte Quellen:")
print("  1. Ergebnis-JSONs: Keine Token-Felder")
print("  2. LangFuse Trace-Observations:")
print("     - input_count: nicht vorhanden")
print("     - output_count: nicht vorhanden")
print("     - total_count: nicht vorhanden")
print("     - usage: nicht vorhanden")
print()
print("Mögliche Lösung für künftige Benchmarks:")
print("  - LangFuse-Integration um Token-Zählung erweitern")
print("  - Oder: Token-Verbrauch separat pro API-Aufruf loggen")


## 9. Diagramm: Task Completion Rate
Balkendiagramm der Erfolgsquote mit Wilson-Score-Konfidenzintervallen (95 %).

In [ ]:
def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple:
    """
    Berechne Wilson-Score-Konfidenzintervall für einen Binomialparameter.
    """
    if total == 0:
        return (0.0, 0.0)
    p_hat = successes / total
    denominator = 1 + z**2 / total
    centre = p_hat + z**2 / (2 * total)
    spread = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * total)) / total)
    return (
        max(0.0, (centre - spread) / denominator),
        min(1.0, (centre + spread) / denominator),
    )

# Completion-Rate-Daten
rate_data = []
for model_short, group in df_with_latency.groupby("model_short"):
    successes = int(group[FIELD_SUCCESS].sum())
    total = len(group)
    rate = successes / total
    ci_lower, ci_upper = wilson_interval(successes, total)
    rate_data.append({
        "Modell": model_short,
        "Erfolge": successes,
        "Gesamt": total,
        "Rate": rate,
        "Rate_%": rate * 100,
        "CI_lower": ci_lower * 100,
        "CI_upper": ci_upper * 100,
    })

rate_df = pd.DataFrame(rate_data)
print("Task Completion Rate")
print("-" * 50)
for _, row in rate_df.iterrows():
    print(f"  {row['Modell']:25s}: {row['Erfolge']}/{row['Gesamt']} = {row['Rate_%']:.1f}% "
          f"[95% CI: {row['CI_lower']:.1f}% – {row['CI_upper']:.1f}%]")

# Balkendiagramm
fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.bar(
    rate_df["Modell"],
    rate_df["Rate_%"],
    yerr=[
        rate_df["Rate_%"] - rate_df["CI_lower"],
        rate_df["CI_upper"] - rate_df["Rate_%"],
    ],
    capsize=6,
    color=sns.color_palette("husl", len(rate_df)),
    edgecolor="black",
    alpha=0.85,
)

# Beschriftung: Erfolge/Versuche + Prozent
for bar, row in zip(bars, rate_df.iterrows()):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 1.5,
        f"{row[1]['Erfolge']}/{row[1]['Gesamt']}\n{height:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.set_title("Task Completion Rate pro Modell", pad=20)
ax.set_xlabel("Modell")
ax.set_ylabel("Erfolgsrate [%]")
ax.set_ylim(0, 115)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_completion.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("\nDiagramm gespeichert: benchmark_completion.png")


## 10. Diagramm: Latenz je Modell, farbkodiert nach Erfolg
Scatterplot der Response Latency gegen das Modell, farbkodiert nach Erfolg/Misserfolg.

In [ ]:
# Scatterplot Latenz vs. Modell, farbkodiert nach Erfolg
fig, ax = plt.subplots(figsize=(12, 7))

palette_success = {True: "#2ecc71", False: "#e74c3c"}
labels_success = {True: "Erfolgreich", False: "Fehlgeschlagen"}

for success_val, color in palette_success.items():
    subset = df_with_latency[df_with_latency[FIELD_SUCCESS] == success_val]
    # Jitter für bessere Sichtbarkeit
    rng = np.random.default_rng(42)
    jitter = rng.uniform(-0.2, 0.2, len(subset))
    ax.scatter(
        subset["model_short"].astype("category").cat.codes + jitter,
        subset["trace_latency_s"],
        c=color,
        alpha=0.6,
        s=80,
        edgecolors="black",
        linewidth=0.5,
        label=labels_success[success_val],
        zorder=3,
    )

# Erfolgsraten als Text einfügen
for model_short, group in df_with_latency.groupby("model_short"):
    rate = group[FIELD_SUCCESS].mean() * 100
    x_pos = df_with_latency["model_short"].astype("category").cat.categories.get_loc(model_short)
    ax.text(
        x_pos, 10, f"{rate:.0f}%",
        ha="center", va="bottom", fontsize=10, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )

ax.set_title("Response Latency je Modell, farbkodiert nach Erfolg", pad=20)
ax.set_xlabel("Modell")
ax.set_ylabel("Latenz [s]")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Ergebnis", loc="upper left")

plt.tight_layout()
plt.savefig("benchmark_latency_by_success.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Diagramm gespeichert: benchmark_latency_by_success.png")


## 11. Zusätzliche Analyse: Latenz getrennt nach Erfolg/Misserfolg

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Links: Latenz nur bei Erfolg
df_success = df_with_latency[df_with_latency[FIELD_SUCCESS] == True]
ax1 = axes[0]
sns.boxplot(
    x="model_short", y="trace_latency_s",
    data=df_success, ax=ax1, palette="Greens",
    width=0.5, fliersize=0,
)
sns.swarmplot(
    x="model_short", y="trace_latency_s",
    data=df_success, ax=ax1, color="black", size=6, alpha=0.6,
)
ax1.set_title("Latenz bei erfolgreicher Task-Abschließung")
ax1.set_ylabel("Latenz [s]")
ax1.set_ylim(bottom=0)
ax1.grid(axis="y", alpha=0.3)

# Rechts: Latenz bei Misserfolg
df_failed = df_with_latency[df_with_latency[FIELD_SUCCESS] == False]
ax2 = axes[1]
sns.boxplot(
    x="model_short", y="trace_latency_s",
    data=df_failed, ax=ax2, palette="Reds",
    width=0.5, fliersize=0,
)
sns.swarmplot(
    x="model_short", y="trace_latency_s",
    data=df_failed, ax=ax2, color="black", size=6, alpha=0.6,
)
ax2.set_title("Latenz bei fehlgeschlagener Task-Abschließung")
ax2.set_ylabel("Latenz [s]")
ax2.set_ylim(bottom=0)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Latenz verglichen: Erfolg vs. Misserfolg", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("benchmark_latency_success_vs_failed.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Diagramm gespeichert: benchmark_latency_success_vs_failed.png")


## 12. Fehleranalyse

In [ ]:
# Fehler-Typen extrahieren
error_df = df[df[FIELD_ERROR].notna()].copy()
error_df["error_type"] = error_df[FIELD_ERROR].apply(
    lambda e: e.get("type", "Unknown") if isinstance(e, dict) else str(e)
)
error_df["error_msg_short"] = error_df[FIELD_ERROR].apply(
    lambda e: e.get("message", "")[:100] if isinstance(e, dict) else str(e)[:100]
)

print("=" * 70)
print("Fehler-Typen pro Modell")
print("=" * 70)

error_counts = error_df.groupby(["model_short", "error_type"]).size().unstack(fill_value=0)
print(error_counts.to_string())
print(f"\nGesamt fehlgeschlagene Läufe: {len(error_df)} / {len(df)} ({len(error_df)/len(df)*100:.1f}%)")

# Fehler-Verteilung als Balkendiagramm
fig, ax = plt.subplots(figsize=(12, 6))
error_pivot = error_df.groupby(["model_short", "error_type"]).size().unstack(fill_value=0)
error_pivot.plot(kind="bar", stacked=True, ax=ax, color=sns.color_palette("husl", len(error_pivot.columns)))
ax.set_title("Fehler-Typen je Modell")
ax.set_xlabel("Modell")
ax.set_ylabel("Anzahl")
ax.legend(title="Fehler-Typ")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("benchmark_error_types.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Diagramm gespeichert: benchmark_error_types.png")


## 13. Ergebniszusammenfassung

---

In [ ]:
print("=" * 70)
print("  BENCHMARK-ZUSAMMENFASSUNG")
print("=" * 70)
print()
print("Rahmenbedingungen")
print(f"  Aufgabe:        Families → Persons Transformation (bxAgent)")
print(f"  Modelle:        {len(MODEL_SHORTCUTS)}")
print(f"  Wiederholungen: 10 pro Modell")
print(f"  Gesamtversuche: {len(df)}")
print()
print("Ergebnisse")
print()
for model_short, group in df_with_latency.groupby("model_short"):
    lat = group["trace_latency_s"]
    succ = int(group[FIELD_SUCCESS].sum())
    total = len(group)
    rate = succ / total * 100
    ci_l, ci_u = wilson_interval(succ, total)
    print(f"  {model_short:30s}: {succ:2d}/{total} = {rate:5.1f}% "
          f"[CI: {ci_l*100:4.1f}% – {ci_u*100:4.1f}%]  |  "
          f"Latenz: {lat.median():6.1f}s (±{lat.std():.1f}s)")

print()
print("Wichtige Hinweise")
print("  - Response Latency aus LangFuse-Trace-Latenz (Start → Ende des Agent-Laufs)")
print("  - Token-Verbrauch in den Daten NICHT verfügbar (JSONs + LangFuse Observations)")
print("  - Erfolgsdefinition: success=True in Ergebnis-JSON")
print("  - Wilson-Konfidenzintervalle (95 %) für kleine Stichproben (n=10)")
print()
print("  Die Auswertung basiert auf 10 Wiederholungen pro Modell und ist als")
print("  erste Eindrücke zu verstehen, nicht als statistisch signifikante Vergleiche.")
print()
print("Schnellstes Modell (Median-Latenz bei Erfolg):")
success_latency = (
    df_with_latency[df_with_latency[FIELD_SUCCESS]]
    .groupby("model_short")["trace_latency_s"]
    .median()
    .sort_values()
)
for model, lat in success_latency.items():
    print(f"  {model:30s}: {lat:.1f}s")

print()
print("Beste Completion Rate:")
best_rate = df_with_latency.groupby("model_short")[FIELD_SUCCESS].mean().max()
for model_short, group in df_with_latency.groupby("model_short"):
    rate = group[FIELD_SUCCESS].mean()
    if rate == best_rate:
        succ = int(group[FIELD_SUCCESS].sum())
        total = len(group)
        print(f"  {model_short:30s}: {succ}/{total} = {rate*100:.1f}%")
print()
print("=" * 70)


## 14. Reproduzierbarkeit
Dieses Notebook ist von oben nach unten ohne manuelle Bearbeitung ausführbar.
Alle Pfade und Zuordnungen sind in Zelle 1 (Konfiguration) zentral definiert.
**Ausführung:** Das Notebook muss im Projektverzeichnis ausgeführt werden, sodass `.mdeagent-benchmark/` im aktuellen Verzeichnis erreichbar ist.
**Benötigte Python-Pakete:**
```bash
pip install pandas numpy matplotlib seaborn scipy langfuse
```